# Задача 2. Рекуррентная сеть для числовой последовательности

2. Сгенерировать последовательности, которые состоят из цифр (от 0 до 9) и задаются следующим образом:

- \(x\) — последовательность цифр;
- \(y_1 = x_1\);
- \(y_i = x_i + x_{i-1}\);
- если \(y_i \ge 10\), то \(y_i = y_i - 10\).

Необходимо научить модель рекуррентной нейронной сети по входной последовательности \(x\) предсказывать соответствующую последовательность \(y\).

Использовать для построения моделей: **RNN, LSTM, GRU**.

**6 баллов** за правильно выполненное задание.

In [1]:
# Блок 0. Импорты

import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [2]:
# Блок 1. Генерация данных по формуле

SEQ_LEN = 10

def generate_pair(seq_len=SEQ_LEN):
    x = []
    y = []
    for i in range(seq_len):
        digit = random.randint(0, 9)
        x.append(digit)
        if i == 0:
            yi = digit
        else:
            yi = digit + x[i - 1]
            if yi >= 10:
                yi -= 10
        y.append(yi)
    return x, y

def generate_dataset(n_samples, seq_len=SEQ_LEN):
    xs = []
    ys = []
    for _ in range(n_samples):
        x, y = generate_pair(seq_len)
        xs.append(x)
        ys.append(y)
    return xs, ys

x_train, y_train = generate_dataset(5000)
x_val,   y_val   = generate_dataset(1000)

print("Пример:", x_train[0], "->", y_train[0])


Пример: [5, 3, 8, 7, 0, 3, 7, 6, 2, 6] -> [5, 8, 1, 5, 7, 3, 0, 3, 8, 8]


In [3]:
# Блок 2. Dataset и DataLoader

class DigitSeqDataset(Dataset):
    def __init__(self, xs, ys):
        self.xs = xs
        self.ys = ys

    def __len__(self):
        return len(self.xs)

    def __getitem__(self, idx):
        x = torch.tensor(self.xs[idx], dtype=torch.long)
        y = torch.tensor(self.ys[idx], dtype=torch.long)
        return x, y

train_ds = DigitSeqDataset(x_train, y_train)
val_ds   = DigitSeqDataset(x_val, y_val)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_dl))
print("Batch shapes:", xb.shape, yb.shape)

Batch shapes: torch.Size([64, 10]) torch.Size([64, 10])


In [4]:
# Блок 3. Модели RNN, LSTM, GRU

class DigitRNN(nn.Module):
    def __init__(self, vocab_size=10, emb_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.RNN(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)
        out, h = self.rnn(emb)
        logits = self.fc(out)
        return logits

class DigitLSTM(nn.Module):
    def __init__(self, vocab_size=10, emb_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)
        out, (h, c) = self.lstm(emb)
        logits = self.fc(out)
        return logits

class DigitGRU(nn.Module):
    def __init__(self, vocab_size=10, emb_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)
        out, h = self.gru(emb)
        logits = self.fc(out)
        return logits

# Можно выбрать любую из трёх:
# model = DigitRNN().to(device)
# model = DigitGRU().to(device)
model = DigitLSTM().to(device)
print(model)

DigitLSTM(
  (embed): Embedding(10, 16)
  (lstm): LSTM(16, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=10, bias=True)
)


In [5]:
# Блок 4. Обучение

def train_digit_model(model, train_dl, val_dl, epochs=8, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(
                logits.view(-1, 10),
                yb.view(-1)
            )
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.inference_mode():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(
                    logits.view(-1, 10),
                    yb.view(-1)
                )
                val_loss += loss.item()
                preds = logits.argmax(dim=-1)
                correct += (preds == yb).sum().item()
                total += yb.numel()

        print(f"Epoch {epoch}: train_loss={train_loss/len(train_dl):.4f}, "
              f"val_loss={val_loss/len(val_dl):.4f}, "
              f"val_token_acc={correct/total:.3f}")

train_digit_model(model, train_dl, val_dl, epochs=8)

Epoch 1: train_loss=2.2819, val_loss=2.2406, val_token_acc=0.189
Epoch 2: train_loss=2.1345, val_loss=1.9640, val_token_acc=0.281
Epoch 3: train_loss=1.7232, val_loss=1.4469, val_token_acc=0.610
Epoch 4: train_loss=1.1859, val_loss=0.9398, val_token_acc=0.854
Epoch 5: train_loss=0.7495, val_loss=0.5849, val_token_acc=0.963
Epoch 6: train_loss=0.4690, val_loss=0.3729, val_token_acc=0.992
Epoch 7: train_loss=0.3050, val_loss=0.2491, val_token_acc=0.999
Epoch 8: train_loss=0.2083, val_loss=0.1752, val_token_acc=1.000


In [6]:
# Блок 5. Примеры предсказаний

def show_examples(model, n_examples=5):
    model.eval()
    for _ in range(n_examples):
        x, y_true = generate_pair()
        x_tensor = torch.tensor(x, dtype=torch.long).unsqueeze(0).to(device)
        with torch.inference_mode():
            logits = model(x_tensor)
        y_pred = logits.argmax(dim=-1).squeeze(0).cpu().tolist()
        print("x      :", x)
        print("y_true :", y_true)
        print("y_pred :", y_pred)
        print("-" * 40)

show_examples(model, n_examples=5)

x      : [9, 7, 8, 5, 8, 1, 4, 2, 9, 8]
y_true : [9, 6, 5, 3, 3, 9, 5, 6, 1, 7]
y_pred : [9, 6, 5, 3, 3, 9, 5, 6, 1, 7]
----------------------------------------
x      : [9, 4, 6, 2, 0, 3, 7, 0, 7, 0]
y_true : [9, 3, 0, 8, 2, 3, 0, 7, 7, 7]
y_pred : [9, 3, 0, 8, 2, 3, 0, 7, 7, 7]
----------------------------------------
x      : [7, 9, 0, 3, 5, 4, 2, 4, 9, 9]
y_true : [7, 6, 9, 3, 8, 9, 6, 6, 3, 8]
y_pred : [7, 6, 9, 3, 8, 9, 6, 6, 3, 8]
----------------------------------------
x      : [7, 1, 5, 6, 4, 3, 8, 8, 7, 8]
y_true : [7, 8, 6, 1, 0, 7, 1, 6, 5, 5]
y_pred : [7, 8, 6, 1, 0, 7, 1, 6, 5, 5]
----------------------------------------
x      : [3, 2, 3, 9, 3, 9, 4, 2, 0, 8]
y_true : [3, 5, 5, 2, 2, 2, 3, 6, 2, 8]
y_pred : [3, 5, 5, 2, 2, 2, 3, 6, 2, 8]
----------------------------------------


## Выводы по задаче 2

- Сгенерирована выборка последовательностей цифр \(x\) длины 10, для которых целевая последовательность \(y\) вычислялась по рекуррентной формуле \(y_1 = x_1\), \(y_i = x_i + x_{i-1}\) с вычитанием 10 при переполнении (работа по модулю 10).
- Для решения задачи many-to-many была построена рекуррентная модель (в данном эксперименте — LSTM), которая на каждом шаге по входному элементу и скрытому состоянию предсказывает соответствующий элемент \(y_i\).
- По логам обучения видно, что функция потерь на обучающей и валидационной выборке монотонно уменьшается, а метрика `val_token_acc` растёт от 0.189 до 1.000 к 8‑й эпохе, что говорит о полной сходимости модели.
- На выведенных примерах тестовых последовательностей значения `y_pred` полностью совпадают с `y_true` для всех позиций, то есть сеть корректно выучила и воспроизводит заданное рекуррентное правило, фактически реализуя вычисление суммы двух соседних цифр по модулю 10.
